# Session 01 · Exercise 2 — Model response via API (OpenAI)

Model: `gpt-4o-mini` · API: Responses API (`client.responses.create`).
Key: Colab Secret `OpenAi-API`. Same question and role as `ex02_anthropic.ipynb`.

In [ ]:
# Instala el SDK de OpenAI
!pip install openai

In [ ]:
import os
from google.colab import userdata

# Cargar la key desde Secrets y asignarla como variable de entorno
os.environ["OPENAI_API_KEY"] = userdata.get("OpenAi-API")

# A partir de aquí, el SDK de OpenAI la detecta automáticamente
from openai import (
    OpenAI,
    AuthenticationError,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    InternalServerError,
)

openai_client = OpenAI()  # No necesitas pasar la key — la lee del entorno

In [ ]:
# gpt-4o-mini pricing (check periodically)
PRICING = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

def query_openai(message, instructions=None, model="gpt-4o-mini", temperature=0.3):
    """
    Make a call to the OpenAI Responses API and return
    the response along with relevant metadata.
    """
    try:
        response = openai_client.responses.create(
            model=model,
            instructions=instructions,
            input=message,
            temperature=temperature,
            max_output_tokens=1000,
            store=False
        )

        # Verify the response is complete
        if response.status != "completed":
            return {"error": f"Response not completed: {response.status}"}

        # Calculate cost
        prices = PRICING.get(model, {"input": 0, "output": 0})
        cost = (
            (response.usage.input_tokens / 1_000_000) * prices["input"] +
            (response.usage.output_tokens / 1_000_000) * prices["output"]
        )

        return {
            "content": response.output_text,
            "model": response.model,
            "id": response.id,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "status": response.status,
            "cost_usd": cost,
        }

    except AuthenticationError:
        return {"error": "Invalid or missing API key"}
    except RateLimitError:
        return {"error": "Rate limit reached or insufficient credit"}
    except BadRequestError as e:
        return {"error": f"Invalid request: {e.message}"}
    except (APIConnectionError, InternalServerError):
        return {"error": "Connection or server error"}


def print_result(title, r):
    """Print the answer and its metadata (Level 3) for one call."""
    print("=" * 60)
    print(title)
    print("=" * 60)
    if "error" in r:
        print(f"Error: {r['error']}")
        return
    print(r["content"])
    print("\n--- Metadata ---")
    print(f"Model: {r['model']}")
    print(f"ID: {r['id']}")
    print(f"Tokens: {r['input_tokens']} input + {r['output_tokens']} output")
    print(f"Status: {r['status']}")
    print(f"Cost: ${r['cost_usd']:.6f}")
    print()

In [ ]:
QUESTION = "How long would it take to build a login page with email and password?"

ROLE = (
    "You are a senior software estimation expert. "
    "Always give an optimistic and a pessimistic estimate in hours, "
    "list the tasks, and state your assumptions."
)

# Level 1: no system prompt
level_1 = query_openai(message=QUESTION)

# Level 2: same question, with a role
level_2 = query_openai(
    message=QUESTION,
    instructions=ROLE,
)

print_result("LEVEL 1 (no role)", level_1)
print_result("LEVEL 2 (with role)", level_2)

## Backup: OpenRouter (not used)

In [ ]:
#from google.colab import userdata
#from openai import OpenAI

# The OpenAI SDK talks to OpenRouter: same request format, different address.
#client = OpenAI(
#    base_url="https://openrouter.ai/api/v1",  # send requests to OpenRouter
#    api_key=userdata.get("OpenRouter"),       # key from Colab Secrets
#)

# Test call. OpenRouter model names use the format "provider/model".
#response = client.chat.completions.create(
#    model="openai/gpt-4o-mini",
#    messages=[{"role": "user", "content": "Say hello in one sentence."}],
#)
#print(response.choices[0].message.content)